# Q16：全部 bull flip 的防守结构研究

保持 Q15 信号、PIT 选池和交易规则冻结，分别研究分市场 MA200 新开仓过滤，以及 RiskParity 核心＋10%/20%/30% 原始 bull flip 卫星；另补做预先指定的核心90%＋MA200 bull10%同口径交叉检验。

In [ ]:
import os
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'xquant').exists():
            return candidate
    raise FileNotFoundError('xquant repository root not found')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'scan-v2-all-bull-flip-defense'


## 1. 实验边界

实验一只改变新开仓资格，已有 bull flip 持仓仍按自身 ST 翻空退出。实验二只改变核心/卫星风险预算。正式 90/10、80/20、70/30 的卫星腿均为原始 bull flip；补充的 core90_def200_bull10 单独标记为两层规则的交叉检验，不能用它外推其他比例。

In [ ]:
import pandas as pd

metrics = pd.read_csv(OUTPUT_DIR / 'portfolio_metrics.csv')
metrics[['strategy', 'cagr', 'annualizedVolatility', 'sharpe', 'calmar',
         'maxDrawdown', 'maxRecoveryDays', 'annualizedTurnover',
         'averageGrossExposure']]

## 2. 年度、月度与压力期

In [ ]:
annual = pd.read_csv(OUTPUT_DIR / 'annual_metrics.csv')
monthly = pd.read_csv(OUTPUT_DIR / 'monthly_distribution.csv')
display(annual.pivot(index='period', columns='strategy', values='return'))
display(monthly)

## 3. 净值、回撤与恢复

In [ ]:
from IPython.display import Image, display

display(Image(filename=OUTPUT_DIR / 'nav_and_drawdown.png'))
display(pd.read_csv(OUTPUT_DIR / 'drawdown_summary.csv'))

## 4. 滚动窗口与核心相关性

In [ ]:
display(pd.read_csv(OUTPUT_DIR / 'rolling_window_summary.csv'))
display(pd.read_csv(OUTPUT_DIR / 'benchmark_comparison.csv'))

## 5. 防守漏斗与参数敏感性

In [ ]:
display(pd.read_csv(OUTPUT_DIR / 'def200_entry_gate_summary.csv'))
display(pd.read_csv(OUTPUT_DIR / 'defense_parameter_cost_sensitivity.csv'))
display(pd.read_csv(OUTPUT_DIR / 'core_volatility_window_sensitivity.csv'))

## 6. 结论

- Historical best：MA200 只暂停新开仓，CAGR 约 21.00%，最大回撤降到约 -21.69%。
- Risk-adjusted best：核心70/bull30 的全样本 Sharpe 最高；补充的核心90/MA200-bull10 的 Calmar 最高。
- Most structurally balanced：bull flip 独立运行优先观察 MA200；保守组合中，核心90/MA200-bull10 的 CAGR 与原始90/10近似，最大回撤从约 -15.03% 降到 -13.49%。
- Main advantage：减少 2018、2022 恶劣环境中的重新入场损失。
- Main failure mode：快速 V 形反弹、市场代理错配、未来趋势机会不足。
- What to test next：冻结原始90/10和MA200-bull90/10做未来纸面并行；80/20、70/30尚未测试MA200卫星，不能直接外推。